### **Sieć neuronowa STL na reprezentacji fingerprint**


Wykorzystana reprezentacja: **ECFP4**

**Lista endpointów**:

1. Caco-2 (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. HIA (Hou)
5. Half Life (Obach)
6. Clearance Hepatocyte (AstraZeneca)
7. CYP3A4 Inhibition (Veith)
8. VDss (Lombardo)
9. AMES Mutagenicity
10. hERG (Wang)

**Metryki jakości**: AUROC (klasyfikacja), RMSE(regresja), Accuracy, F1 (klasyfikacja), R^2(regresja), MAE(regresja)

In [ ]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 26.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# co jest w MyDrive w ogóle
print("=== /content/drive/MyDrive/ ===")
for f in sorted(os.listdir('/content/drive/MyDrive/')):
    print(f"  {f}")

# sprawdź czy istnieje "MLDD - ADMET" i zajrzyj do środka
admet_path = '/content/drive/MyDrive/MLDD - ADMET'
if os.path.exists(admet_path):
    print(f"\n=== {admet_path} ===")
    for f in sorted(os.listdir(admet_path)):
        print(f"  {f}")
else:
    print(f"\n✗ {admet_path} — NIE istnieje")

In [ ]:
data_folder = "/content/drive/MyDrive/mldd_data" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

# DEFINICJA SIECI NEURONOWEJ

class AdmetEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.res_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.main(x)
        return torch.relu(x + self.res_layer(x))

# --- STL REGRESOR ---
class STL_NN_Regressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1) # Wynik liniowy (dowolna liczba)

    def forward(self, x):
        return self.head(self.encoder(x))

# --- STL KLASYFIKATOR ---
class STL_NN_Classifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Sigmoid zamienia wynik na prawdopodobieństwo (0-1)
        return torch.sigmoid(self.head(self.encoder(x)))

  #=============================

class Featurizer:
    def __init__(self, y_column, smiles_col='Drug', **kwargs):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.__dict__.update(kwargs)
    def __call__(self, df):
        raise NotImplementedError()

class ECFPFeaturizer(Featurizer):
    def __init__(self, y_column, radius=2, length=1024, **kwargs):
        self.radius = radius
        self.length = length
        super().__init__(y_column, **kwargs)

    def __call__(self, df):
        fingerprints = []
        labels = []
        for i, row in df.iterrows():
            y = row[self.y_column]
            smiles = row[self.smiles_col]
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.length)
                fingerprints.append(np.array(fp))
                labels.append(y)
        return np.array(fingerprints), np.array(labels).reshape(-1, 1)



In [ ]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [ ]:
def print_metrics(metrics, task='classification'):
    print(f"\n{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification'):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np):

    # Normalizacja etykiet
    print("Normalizacja danych...")
    scaler = StandardScaler()
    y_train_scaled = scaler.fit_transform(y_train_np)
    y_test_scaled = scaler.transform(y_test_np)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.FloatTensor(X_train_np).to(device)
    y_train_t = torch.FloatTensor(y_train_scaled).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = STL_NN_Regressor(input_dim=1024).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("\n--- START TRENINGU (Regresja) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            mse_val = loss.item()
            print(f"  Epoka {epoch:3d} | Błąd MSE: {loss.item():.4f}")

# Ewaluacja
    print("\n--- EWALUACJA ---")
    model.eval()
    with torch.no_grad():
        preds_scaled = model(X_test_t).cpu().numpy()

    preds = scaler.inverse_transform(preds_scaled)

    mse = mean_squared_error(y_test_np, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_np, preds)
    r2 = r2_score(y_test_np, preds)

    metrics = {
        "test_metrics": {
            "rmse": rmse,
            "mae": mae,
            "r2": r2
        }
    }

    return metrics


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
def train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np):

#dla klasyfikacji nie normalizujemy

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.FloatTensor(X_train_np).to(device)
    y_train_t = torch.FloatTensor(y_train_np).to(device) # Etykiety 0.0 lub 1.0
    X_test_t = torch.FloatTensor(X_test_np).to(device)

# Model i Trening
    model = STL_NN_Classifier(input_dim=1024).to(device)
    criterion = nn.BCELoss() # Binary Cross Entropy
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("\n--- START TRENINGU (Klasyfikacja) ---")
    model.train()
    for epoch in range(101):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Błąd BCE Loss: {loss.item():.4f}")

# Ewaluacja
    print("\n--- EWALUACJA ---")
    model.eval()
    with torch.no_grad():
        # Model zwraca prawdopodobieństwo (0 do 1)
        probs = model(X_test_t).cpu().numpy()
        # Klasy (0 lub 1) ustalamy progiem 0.5
    preds = (probs > 0.5).astype(int)

    auroc = roc_auc_score(y_test_np, probs)
    acc = accuracy_score(y_test_np, preds)

    print(f"Wynik końcowy AUROC (HIA): {auroc:.4f}")
    print(f"Dokładność (Accuracy): {acc*100:.2f}%")

    metrics = {
        "test_metrics": {
            "accuracy": accuracy_score(y_test_np, preds),
            "f1":       f1_score(y_test_np, preds),
            "auroc":    roc_auc_score(y_test_np, probs),
        }
    }

    return metrics

Endpoint 1: Caco-2 (Wang)

In [ ]:
train, test = load_split_pickle('Caco2_Wang')

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

X_train_np, y_train_np = featurizer(train)
print(y_train_np)
X_test_np, y_test_np = featurizer(test)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Caco2_Wang", "metrics_NN_fingerprints.txt", task="regression")

Endpoint 2:  Lipophilicity

In [ ]:
train, test = load_split_pickle('Lipophilicity_AstraZeneca')

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

X_train_np, y_train_np = featurizer(train)
X_test_np, y_test_np = featurizer(test)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Lipophilicity_AstraZeneca", "metrics_NN_fingerprints.txt", task="regression")

Endpoint 3: Solubility

In [ ]:
train, test = load_split_pickle('Solubility_AqSolDB')

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

X_train_np, y_train_np = featurizer(train)
X_test_np, y_test_np = featurizer(test)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Solubility_AqSolDB", "metrics_NN_fingerprints.txt", task="regression")

Endpoint 4: HIA

In [ ]:
train, test = load_split_pickle('HIA_Hou')

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

X_train_np, y_train_np = featurizer(train)
X_test_np, y_test_np = featurizer(test)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "HIA_Hou", "metrics_NN_fingerprints.txt")

Endpoint 5: Half Life

In [ ]:
train, test = load_split_pickle('Half_Life_Obach')

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

X_train_np, y_train_np = featurizer(train)
X_test_np, y_test_np = featurizer(test)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Half_Life_Obach", "metrics_NN_fingerprints.txt", task="regression")

Endpoint 6: Clearance Hepatocyte

In [ ]:
train, test = load_split_pickle('Clearance_Hepatocyte_AZ')

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

X_train_np, y_train_np = featurizer(train)
X_test_np, y_test_np = featurizer(test)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Clearance_Hepatocyte_AZ", "metrics_NN_fingerprints.txt", task="regression")

Endpoint 7: CYP3A4 Inhibition

In [ ]:
train, test = load_split_pickle('CYP3A4_Veith')

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

X_train_np, y_train_np = featurizer(train)
X_test_np, y_test_np = featurizer(test)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "CYP3A4_Veith", "metrics_NN_fingerprints.txt")

Endpoint 8: VDss

In [ ]:
train, test = load_split_pickle('VDss_Lombardo')

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

X_train_np, y_train_np = featurizer(train)
X_test_np, y_test_np = featurizer(test)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "VDss_Lombardo", "metrics_NN_fingerprints.txt", task="regression")

Endpoint 9: AMES Mutagenicity

In [ ]:
train, test = load_split_pickle('AMES')

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

X_train_np, y_train_np = featurizer(train)
X_test_np, y_test_np = featurizer(test)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "AMES", "metrics_NN_fingerprints.txt")

Endpoint 10: hERG (Wang)

In [ ]:
train, test = load_split_pickle('hERG')

featurizer = ECFPFeaturizer(y_column='Y', length=1024)

X_train_np, y_train_np = featurizer(train)
X_test_np, y_test_np = featurizer(test)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "hERG", "metrics_NN_fingerprints.txt")